# 📚 Análise de Desempenho Escolar no Brasil
**Projeto G2 — Tema 24**  
Disciplina: Linguagem de Programação — Análise e Visualização de Dados com Python

---

## 1. Introdução ao Problema

O desempenho escolar é um dos principais indicadores da qualidade educacional de um país e está diretamente relacionado ao desenvolvimento humano, à redução das desigualdades sociais e às oportunidades profissionais futuras dos estudantes.

No Brasil, um país de dimensões continentais e realidades socioeconômicas muito distintas entre regiões, compreender os padrões de desempenho escolar é essencial para a formulação de políticas públicas eficazes e para a identificação de fatores que favorecem ou prejudicam o aprendizado.

Este projeto realiza uma análise exploratória completa de dados educacionais brasileiros entre **2015 e 2024**, buscando responder às seguintes perguntas orientadoras:

1. Quais estados apresentam melhor desempenho escolar?
2. Existem diferenças entre escolas públicas e privadas?
3. Houve melhoria no desempenho ao longo do tempo?
4. Existe relação entre renda familiar e desempenho?
5. Como o acesso à internet impacta o aprendizado?
6. Quais disciplinas apresentam menor rendimento?
7. Existem desigualdades regionais relevantes?

## 2. Contextualização Educacional

O sistema educacional brasileiro é composto por escolas públicas (federais, estaduais e municipais) e privadas, distribuídas nas cinco grandes regiões do país: **Norte, Nordeste, Centro-Oeste, Sudeste e Sul**. Historicamente, há uma grande disparidade de desempenho entre essas regiões, influenciada por fatores como:

- **Renda familiar:** famílias com maior renda tendem a ter mais acesso a recursos educacionais complementares;
- **Acesso à tecnologia:** o acesso à internet permite o uso de ferramentas digitais de aprendizado;
- **Rede de ensino:** escolas privadas costumam ter mais recursos pedagógicos e infraestrutura;
- **Localização geográfica:** regiões com menor índice de desenvolvimento humano frequentemente apresentam desempenho educacional mais baixo.

Esta análise utiliza um dataset simulado que reflete padrões realistas do cenário educacional brasileiro.

## 3. Explicação da Base de Dados

O dataset utilizado é o `simulacao_desempenho_escolar_brasil.csv`, contendo registros de desempenho escolar por região, estado, município, rede de ensino e disciplina, cobrindo o período de 2015 a 2024.

| Coluna | Tipo | Descrição |
|---|---|---|
| ano | int | Ano letivo (2015–2024) |
| semestre | int | Semestre letivo (1 ou 2) |
| data | str | Data de referência |
| regiao | str | Região do Brasil |
| uf | str | Sigla do estado |
| municipio | str | Nome do município |
| rede_ensino | str | Pública ou Privada |
| disciplina | str | Disciplina avaliada |
| media_notas | float | Média das notas (0–100) |
| taxa_aprovacao | float | Percentual de aprovação |
| taxa_reprovacao | float | Percentual de reprovação |
| acesso_internet | float | Percentual com acesso à internet |
| renda_media_familiar | float | Renda média familiar (R$) |
| indice_desempenho | float | Índice geral de desempenho |
| nivel_desempenho | str | Baixo, Médio, Alto ou Excelente |

## 4. Leitura dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='Blues_d')

# Leitura do dataset
df = pd.read_csv('../dados/simulacao_desempenho_escolar_brasil.csv')

print(f'Shape: {df.shape}')
print(f'Período: {df["ano"].min()} a {df["ano"].max()}')
df.head(10)

In [ ]:
# Informações gerais
df.info()

In [ ]:
# Estatísticas descritivas
df.describe().round(2)

## 5. Limpeza e Preparação

In [ ]:
# Verificação de valores nulos
print('=== Valores Nulos ===')
print(df.isnull().sum())
print(f'\nTotal de nulos: {df.isnull().sum().sum()}')

In [ ]:
# Verificação de duplicatas
print(f'Registros duplicados: {df.duplicated().sum()}')

# Converter coluna data para datetime
df['data'] = pd.to_datetime(df['data'], errors='coerce')

# Verificar valores únicos das colunas categóricas
print('\n=== Valores Únicos ===')
categoricas = ['regiao', 'rede_ensino', 'disciplina', 'nivel_desempenho']
for col in categoricas:
    print(f'{col}: {sorted(df[col].unique().tolist())}')

In [ ]:
# Verificar ranges das colunas numéricas
numericas = ['media_notas', 'taxa_aprovacao', 'taxa_reprovacao',
             'acesso_internet', 'renda_media_familiar', 'indice_desempenho']

print('=== Ranges das colunas numéricas ===')
for col in numericas:
    print(f'{col}: min={df[col].min():.2f} | max={df[col].max():.2f} | média={df[col].mean():.2f}')

print('\nDataset limpo e pronto para análise!')

## 6. Engenharia de Atributos

In [ ]:
# Criar coluna período (ano + semestre)
df['periodo'] = df['ano'].astype(str) + '/S' + df['semestre'].astype(str)

# Ordenar o nivel_desempenho como variável ordinal
ordem_nivel = ['Baixo', 'Médio', 'Alto', 'Excelente']
df['nivel_desempenho'] = pd.Categorical(
    df['nivel_desempenho'],
    categories=ordem_nivel,
    ordered=True
)

# Faixas de renda familiar
df['faixa_renda'] = pd.cut(
    df['renda_media_familiar'],
    bins=[0, 2000, 3000, 4500, 10000],
    labels=['Baixa (até R$2.000)', 'Média-baixa (R$2k–3k)',
            'Média-alta (R$3k–4,5k)', 'Alta (acima R$4,5k)']
)

# Faixas de acesso à internet
df['faixa_internet'] = pd.cut(
    df['acesso_internet'],
    bins=[0, 40, 60, 80, 100],
    labels=['Baixo (0–40%)', 'Médio (40–60%)', 'Alto (60–80%)', 'Muito alto (80–100%)']
)

print('Novas colunas criadas:')
print(df[['periodo', 'nivel_desempenho', 'faixa_renda', 'faixa_internet']].head(8))

## 7. KPIs — Indicadores-Chave de Desempenho

In [ ]:
# Cálculo dos KPIs
kpi_media_notas  = df['media_notas'].mean()
kpi_taxa_aprov   = df['taxa_aprovacao'].mean()
kpi_taxa_reprov  = df['taxa_reprovacao'].mean()
kpi_idx_desemp   = df['indice_desempenho'].mean()

kpi_melhor_uf    = df.groupby('uf')['media_notas'].mean().idxmax()
kpi_melhor_uf_v  = df.groupby('uf')['media_notas'].mean().max()

kpi_pior_disc    = df.groupby('disciplina')['media_notas'].mean().idxmin()
kpi_pior_disc_v  = df.groupby('disciplina')['media_notas'].mean().min()

kpi_melhor_rede  = df.groupby('rede_ensino')['media_notas'].mean().idxmax()
kpi_melhor_rede_v= df.groupby('rede_ensino')['media_notas'].mean().max()

print('=' * 55)
print('           KPIs — DESEMPENHO ESCOLAR NO BRASIL')
print('=' * 55)
print(f'  Média geral das notas         : {kpi_media_notas:.2f}')
print(f'  Taxa média de aprovação       : {kpi_taxa_aprov:.2f}%')
print(f'  Taxa média de reprovação      : {kpi_taxa_reprov:.2f}%')
print(f'  Índice médio de desempenho    : {kpi_idx_desemp:.2f}')
print(f'  Estado com melhor desempenho  : {kpi_melhor_uf} ({kpi_melhor_uf_v:.2f})')
print(f'  Disciplina mais crítica       : {kpi_pior_disc} ({kpi_pior_disc_v:.2f})')
print(f'  Rede com maior desempenho     : {kpi_melhor_rede} ({kpi_melhor_rede_v:.2f})')
print('=' * 55)

## 8. Visualizações

### 8.1 Evolução Temporal do Desempenho

In [ ]:
evolucao = df.groupby('ano')[['media_notas', 'taxa_aprovacao', 'indice_desempenho']].mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Evolução Temporal do Desempenho Escolar (2015–2024)', fontsize=14, fontweight='bold', y=1.02)

cores = ['#2e6da4', '#27ae60', '#e67e22']
cols  = ['media_notas', 'taxa_aprovacao', 'indice_desempenho']
titles= ['Média das Notas', 'Taxa de Aprovação (%)', 'Índice de Desempenho']

for ax, col, title, cor in zip(axes, cols, titles, cores):
    ax.plot(evolucao['ano'], evolucao[col], marker='o', color=cor, linewidth=2.5, markersize=6)
    ax.fill_between(evolucao['ano'], evolucao[col], alpha=0.12, color=cor)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Ano')
    ax.set_xticks(evolucao['ano'])
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../imagens/evolucao_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.2 Desempenho por Região

In [ ]:
por_regiao = df.groupby('regiao')[['media_notas', 'taxa_aprovacao', 'indice_desempenho']].mean().reset_index()
por_regiao = por_regiao.sort_values('media_notas', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Desempenho por Região do Brasil', fontsize=14, fontweight='bold')

for ax, col, title, cor in zip(axes, cols, titles, cores):
    bars = ax.barh(por_regiao['regiao'], por_regiao[col], color=cor, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(title)
    ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../imagens/desempenho_por_regiao.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.3 Desempenho por Estado

In [ ]:
por_uf = df.groupby('uf')['media_notas'].mean().reset_index().sort_values('media_notas', ascending=True)

fig, ax = plt.subplots(figsize=(14, 9))
colors = ['#c0392b' if v < 60 else '#2e6da4' for v in por_uf['media_notas']]
bars = ax.barh(por_uf['uf'], por_uf['media_notas'], color=colors, alpha=0.85, edgecolor='white')
ax.axvline(x=por_uf['media_notas'].mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Média nacional ({por_uf["media_notas"].mean():.1f})')
ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=9)
ax.set_title('Média das Notas por Estado', fontsize=14, fontweight='bold')
ax.set_xlabel('Média das Notas')
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../imagens/desempenho_por_estado.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.4 Pública vs Privada

In [ ]:
por_rede = df.groupby(['ano', 'rede_ensino'])['media_notas'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for rede, cor in [('Pública', '#2e6da4'), ('Privada', '#e67e22')]:
    dados = por_rede[por_rede['rede_ensino'] == rede]
    ax.plot(dados['ano'], dados['media_notas'], marker='o', label=rede,
            color=cor, linewidth=2.5, markersize=7)
    ax.fill_between(dados['ano'], dados['media_notas'], alpha=0.1, color=cor)

ax.set_title('Evolução da Média das Notas: Pública vs Privada', fontsize=13, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Média das Notas')
ax.set_xticks(por_rede['ano'].unique())
ax.legend(fontsize=11)
ax.tick_params(axis='x', rotation=45)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../imagens/publica_vs_privada.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.5 Desempenho por Disciplina

In [ ]:
por_disc = df.groupby('disciplina')[['media_notas', 'taxa_aprovacao']].mean().reset_index()
por_disc = por_disc.sort_values('media_notas', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Desempenho por Disciplina', fontsize=14, fontweight='bold')

palette = sns.color_palette('Blues_d', len(por_disc))

bars1 = axes[0].bar(por_disc['disciplina'], por_disc['media_notas'], color=palette, edgecolor='white')
axes[0].bar_label(bars1, fmt='%.1f', padding=4, fontsize=10)
axes[0].set_title('Média das Notas por Disciplina', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Disciplina')
axes[0].set_ylabel('Média das Notas')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

bars2 = axes[1].bar(por_disc['disciplina'], por_disc['taxa_aprovacao'], color=palette, edgecolor='white')
axes[1].bar_label(bars2, fmt='%.1f%%', padding=4, fontsize=10)
axes[1].set_title('Taxa de Aprovação por Disciplina', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Disciplina')
axes[1].set_ylabel('Taxa de Aprovação (%)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../imagens/desempenho_por_disciplina.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.6 Heatmap Semestral

In [ ]:
pivot_heat = df.pivot_table(
    index='regiao', columns=['ano', 'semestre'],
    values='media_notas', aggfunc='mean'
).round(1)

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(
    pivot_heat, annot=True, fmt='.1f',
    cmap='YlOrRd', linewidths=0.5,
    linecolor='white', annot_kws={'size': 8},
    ax=ax
)
ax.set_title('Heatmap — Média das Notas por Região e Período (Ano/Semestre)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Ano / Semestre')
ax.set_ylabel('Região')
plt.tight_layout()
plt.savefig('../imagens/heatmap_semestral.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.7 Correlação: Renda x Notas e Internet x Notas

In [ ]:
corr_renda = df[['renda_media_familiar', 'media_notas']].corr().iloc[0, 1]
corr_inet  = df[['acesso_internet', 'media_notas']].corr().iloc[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análise de Correlação — Fatores Socioeconômicos e Desempenho', fontsize=13, fontweight='bold')

# Dispersão: Renda x Notas
axes[0].scatter(df['renda_media_familiar'], df['media_notas'],
                alpha=0.4, color='#2e6da4', s=20, edgecolors='none')
m1, b1 = np.polyfit(df['renda_media_familiar'], df['media_notas'], 1)
x1 = np.linspace(df['renda_media_familiar'].min(), df['renda_media_familiar'].max(), 100)
axes[0].plot(x1, m1*x1 + b1, color='#c0392b', linewidth=2, label=f'Correlação: {corr_renda:.4f}')
axes[0].set_title('Renda Familiar x Média das Notas', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Renda Média Familiar (R$)')
axes[0].set_ylabel('Média das Notas')
axes[0].legend(fontsize=10)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Dispersão: Internet x Notas
axes[1].scatter(df['acesso_internet'], df['media_notas'],
                alpha=0.4, color='#27ae60', s=20, edgecolors='none')
m2, b2 = np.polyfit(df['acesso_internet'], df['media_notas'], 1)
x2 = np.linspace(df['acesso_internet'].min(), df['acesso_internet'].max(), 100)
axes[1].plot(x2, m2*x2 + b2, color='#c0392b', linewidth=2, label=f'Correlação: {corr_inet:.4f}')
axes[1].set_title('Acesso à Internet x Média das Notas', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Acesso à Internet (%)')
axes[1].set_ylabel('Média das Notas')
axes[1].legend(fontsize=10)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../imagens/correlacoes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Correlação Renda x Notas  : {corr_renda:.4f}')
print(f'Correlação Internet x Notas: {corr_inet:.4f}')

### 8.8 Matriz de Correlação Geral

In [ ]:
numericas = ['media_notas', 'taxa_aprovacao', 'taxa_reprovacao',
             'acesso_internet', 'renda_media_familiar', 'indice_desempenho']

corr_matrix = df[numericas].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.3f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 10}, ax=ax
)
ax.set_title('Matriz de Correlação — Variáveis Numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../imagens/matriz_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.9 Distribuição dos Níveis de Desempenho por Região

In [ ]:
nivel_regiao = df.groupby(['regiao', 'nivel_desempenho'], observed=True).size().reset_index(name='contagem')
nivel_pct    = nivel_regiao.copy()
total        = nivel_regiao.groupby('regiao')['contagem'].transform('sum')
nivel_pct['pct'] = (nivel_regiao['contagem'] / total * 100).round(1)

pivot_nivel = nivel_pct.pivot(index='regiao', columns='nivel_desempenho', values='pct').fillna(0)
pivot_nivel = pivot_nivel[['Baixo', 'Médio', 'Alto', 'Excelente']]

cores_nivel = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
pivot_nivel.plot(kind='bar', stacked=True, color=cores_nivel,
                 figsize=(12, 6), edgecolor='white', linewidth=0.5)
plt.title('Distribuição dos Níveis de Desempenho por Região (%)', fontsize=13, fontweight='bold')
plt.xlabel('Região')
plt.ylabel('Percentual (%)')
plt.legend(title='Nível', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=0)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../imagens/niveis_por_regiao.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Interpretação dos Resultados

A análise dos dados revelou os seguintes padrões principais:

### 9.1 Desempenho Geral
- A **média geral das notas** no período foi de **64,97**, o que indica um nível de desempenho **moderado** na escala de 0 a 100.
- A **taxa de aprovação média** de **77,42%** demonstra que cerca de **1 em cada 4 alunos** enfrenta dificuldades em ser aprovado.
- O **índice de desempenho médio** de **64,86** é consistente com as demais métricas.

### 9.2 Desigualdades Regionais
- O estado de **MT (Mato Grosso)** apresenta a maior média de notas (**71,75**), enquanto outros estados da região Norte e Nordeste tendem a performar abaixo da média nacional.
- A linha tracejada do gráfico por estado permite identificar claramente os estados que ficam **abaixo da média nacional**.

### 9.3 Disciplinas
- **Português** é a disciplina com **menor rendimento médio** (**63,13**), sendo a mais crítica do sistema.
- As demais disciplinas (Matemática, História, Geografia e Ciências) apresentam médias próximas entre si.

### 9.4 Rede de Ensino
- A rede **Pública** apresenta desempenho ligeiramente superior à Privada neste dataset — o que pode refletir características específicas do dataset simulado.
- A evolução temporal mostra trajetórias próximas entre as redes.

### 9.5 Fatores Socioeconômicos
- A **correlação entre renda familiar e notas** é de **0,0069**, praticamente nula — indicando que, neste dataset, a renda não é um preditor linear significativo do desempenho.
- Da mesma forma, a **correlação entre acesso à internet e notas** é de **0,0055**, também próxima de zero.
- Isso pode sugerir que o impacto desses fatores é mais **não-linear ou mediado por outras variáveis**, ou que o dataset simulado não captura completamente essas relações.

## 10. Conclusão

Este projeto realizou uma análise exploratória abrangente do desempenho escolar no Brasil entre 2015 e 2024, utilizando um dataset com 740 registros e 15 variáveis.

### Principais achados:

1. **O desempenho escolar nacional é moderado**, com média de notas próxima a 65 em 100 e taxa de aprovação de 77%.
2. **Há desigualdades regionais e estaduais relevantes**, com estados do Centro-Oeste (especialmente MT) apresentando os melhores índices.
3. **Português é a disciplina mais crítica**, exigindo atenção especial de políticas educacionais.
4. **A rede de ensino apresenta diferenças menores do que esperado**, sugerindo que outros fatores podem ser mais determinantes para o desempenho.
5. **Os fatores socioeconômicos analisados** (renda e internet) não apresentaram correlação linear expressiva com as notas neste dataset simulado.

### Recomendações:
- Concentrar esforços pedagógicos no ensino de Língua Portuguesa;
- Investigar as boas práticas dos estados com melhor desempenho para replicá-las;
- Ampliar o investimento em infraestrutura digital nas regiões Norte e Nordeste;
- Monitorar continuamente os indicadores ao longo dos semestres para detectar tendências.

---
*Análise desenvolvida como parte do Projeto G2 — Tema 24 | Disciplina: Linguagem de Programação — Análise e Visualização de Dados com Python*